# **🌲 Forest Segmentation with U-Net**
---
In this lab, we will:
✅ **Understand the U-Net architecture** for semantic segmentation  
✅ **Load the Forest Aerial dataset** (images + masks)  
✅ **Build a U-Net from scratch** in PyTorch  
✅ **Train it** to segment forest vs. non-forest pixels  
✅ **Evaluate** with Dice / IoU and **visualize predictions**  

---

## **1️⃣ What is Segmentation?**
Unlike **classification** (one label per image) or **object detection** (a box per object),
**semantic segmentation** assigns a class to **every single pixel**. 🎯

For our forest dataset this is a **binary** problem:

| Pixel value | Meaning |
|-------------|---------|
| **1 (white)** | 🌳 Forest |
| **0 (black)** | 🚫 Not forest |

So for each aerial image, the model outputs a **mask** the same size as the input — a per-pixel "forest / not forest" decision.

---

## **2️⃣ The U-Net Architecture**
U-Net is shaped like the letter **U** — hence the name. It has three parts:

🔽 **Encoder (contracting path)** → repeated *conv → conv → downsample*. Captures **context** ("what" is in the image) while shrinking spatial size.  
🔼 **Decoder (expanding path)** → repeated *upsample → conv → conv*. Recovers **location** ("where" things are) back to full resolution.  
🔗 **Skip connections** → copy feature maps from the encoder straight across to the decoder. This is the *key idea* — it lets the decoder reuse fine spatial detail that would otherwise be lost during downsampling.

```
 input ─► [enc1] ───────────────skip──────────────► [dec1] ─► output
            │                                          ▲
            ▼                                          │
          [enc2] ──────────skip─────────► [dec2] ──────┘
            │                                ▲
            ▼                                │
          [enc3] ────skip────► [dec3] ───────┘
            │                     ▲
            ▼                     │
            └──────► [bottleneck]─┘
```

### 🧩 We won't hand-code it — we'll use `segmentation_models_pytorch`
Writing U-Net from scratch is a great learning exercise, but in practice everyone uses the
**`segmentation_models_pytorch`** (**smp**) library. It gives you a battle-tested U-Net where you can
**plug in any encoder** — including **ImageNet-pretrained** backbones — with a single argument:

```python
model = smp.Unet(encoder_name="resnet34", encoder_weights="imagenet", ...)
```

That `encoder_name` is the interesting knob, and we'll explain it in detail below. 👇

---

#### Step 0 — (Colab only) Set the KaggleHub cache directory

In [ ]:
# # Uncomment when running on Google Colab
# import os
# os.environ["KAGGLEHUB_CACHE"] = "/content/data"

## **3️⃣ Imports & Setup**

In [ ]:
# Install segmentation-models-pytorch (smp)
# !pip install -q segmentation-models-pytorch

In [ ]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("smp version:", smp.__version__)

## **4️⃣ Downloading the Dataset**
The dataset ships as two parallel folders — **`images/`** and **`masks/`** — where each image
has a mask **with the same filename**. A `meta_data.csv` also lists the pairs.

In [ ]:
import kagglehub

# Download the dataset
path = kagglehub.dataset_download("quadeer15sh/augmented-forest-segmentation")
print("Path to dataset files:", path)

In [ ]:
# Explore the folder structure
for root, dirs, files in os.walk(path):
    depth = root.replace(path, "").count(os.sep)
    indent = "    " * depth
    print(f"{indent}📂 {os.path.basename(root)}/  ({len(files)} files)")
    if files:
        print(f"{indent}    e.g. {files[0]}")
    if depth >= 2:
        dirs[:] = []

In [ ]:
# Locate the images/ and masks/ folders (paths can vary slightly by dataset version)
image_dir = glob.glob(os.path.join(path, "**", "images"), recursive=True)[0]
mask_dir  = glob.glob(os.path.join(path, "**", "masks"),  recursive=True)[0]

print("Images:", image_dir)
print("Masks :", mask_dir)

image_paths = sorted(glob.glob(os.path.join(image_dir, "*.jpg")))
mask_paths  = sorted(glob.glob(os.path.join(mask_dir,  "*.jpg")))
print(f"\nFound {len(image_paths)} images and {len(mask_paths)} masks")

#### Let's look at an image and its mask side-by-side 👀

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    img  = Image.open(image_paths[i]).convert("RGB")
    mask = Image.open(mask_paths[i]).convert("L")   # grayscale mask
    axes[0, i].imshow(img);  axes[0, i].set_title("Image"); axes[0, i].axis("off")
    axes[1, i].imshow(mask, cmap="gray"); axes[1, i].set_title("Mask (🌳 = white)"); axes[1, i].axis("off")
plt.tight_layout()
plt.show()

## **5️⃣ Building the `Dataset`**
Our custom `Dataset` returns an **(image, mask)** pair for each index. A few important details for segmentation:

- The **image** is normalized like a normal RGB tensor.  
- The **mask** is **not** normalized — we threshold it to **{0, 1}** and keep it as a float target.  
- Image and mask are resized to the **same** `IMG_SIZE` so they stay pixel-aligned.

In [ ]:
IMG_SIZE = 256   # resize everything to 256x256 (keeps training fast)

# Pretrained encoders were trained on ImageNet, so we normalize with ImageNet stats.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

image_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),                                  # -> [0,1], shape (3,H,W)
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),      # match the pretrained encoder
])

mask_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),                                  # -> [0,1], shape (1,H,W)
])


class ForestDataset(Dataset):
    def __init__(self, image_paths, mask_paths):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img  = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx]).convert("L")

        img  = image_tf(img)
        mask = mask_tf(mask)

        # Binarize the mask: forest = 1, background = 0
        mask = (mask > 0.5).float()
        return img, mask


dataset = ForestDataset(image_paths, mask_paths)
print("Dataset size:", len(dataset))

img, mask = dataset[0]
print("Image tensor:", img.shape, "| Mask tensor:", mask.shape, "| Mask values:", mask.unique())

## **6️⃣ Train / Validation Split & DataLoaders**

In [ ]:
val_frac = 0.15
val_size = int(len(dataset) * val_frac)
train_size = len(dataset) - val_size

train_ds, val_ds = random_split(
    dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)   # reproducible split
)

BATCH_SIZE = 16
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {train_size} images | Val: {val_size} images")

imgs, masks = next(iter(train_loader))
print("Batch:", imgs.shape, masks.shape)

## **7️⃣ Building U-Net with `smp` — and Choosing an Encoder**

With smp, the whole model is **one line**. The decoder is always the U-Net expanding path;
what changes is the **encoder** (a.k.a. the **backbone**) — the network that does the downsampling
and feature extraction on the way down the "U".

```python
model = smp.Unet(
    encoder_name="resnet34",      # <-- the backbone (this is the knob we care about)
    encoder_weights="imagenet",   # start from ImageNet-pretrained weights
    in_channels=3,                # RGB input
    classes=1,                    # 1 output channel -> binary forest mask
)
```

### 🧠 What is the encoder, really?
The encoder is just a **classification network with its head chopped off**. Instead of ending in a
1000-class prediction, smp taps into the encoder's **intermediate feature maps** at several depths
and feeds them into the U-Net decoder via the **skip connections**. Because these backbones are
**pretrained on ImageNet**, they already know edges, textures, and shapes — so segmentation
converges **faster** and to **higher accuracy** than training from scratch. 🎯

### 🔬 Common encoder families
| `encoder_name` | Family | Params | Vibe |
|----------------|--------|--------|------|
| `resnet18` | ResNet | ~11M | Fast, light, great baseline |
| `resnet34` | ResNet | ~21M | The sweet spot for many tasks ⭐ |
| `resnet50` | ResNet | ~23M | Deeper, stronger, a bit slower |
| `efficientnet-b0` | EfficientNet | ~5M | Very parameter-efficient |
| `efficientnet-b3` | EfficientNet | ~12M | Strong accuracy/size trade-off |
| `mobilenet_v2` | MobileNet | ~3M | Tiny & fast (edge / real-time) |
| `timm-regnetx_040` | RegNet | ~22M | Modern, well-regularized |

### ⚖️ How to pick
- 🏃 **Limited GPU / need speed?** → `resnet18`, `mobilenet_v2`, `efficientnet-b0`  
- ⭐ **Good default?** → `resnet34` — cheap, reliable, hard to beat  
- 🎯 **Chasing max accuracy?** → `resnet50`, `efficientnet-b3` (watch memory & time)  

> 💡 **The whole point:** switching encoders is a **one-word change** (`encoder_name=...`). That's why
> smp is so handy for experiments — you can benchmark five backbones without rewriting the model.

> ⚠️ **Two things to keep consistent when you swap encoders:**
> 1. Keep `encoder_weights="imagenet"` **and** normalize inputs with **ImageNet mean/std** (we did this in the dataset).
> 2. Bigger encoders need **more GPU memory** — if you hit OOM, lower `BATCH_SIZE` or `IMG_SIZE`.

In [ ]:
# Pick your encoder here — try swapping this one word later!
ENCODER = "resnet34"

def build_model(encoder_name=ENCODER):
    """Create a U-Net with the chosen (ImageNet-pretrained) encoder."""
    return smp.Unet(
        encoder_name=encoder_name,
        encoder_weights="imagenet",   # pretrained backbone
        in_channels=3,                # RGB
        classes=1,                    # single-channel forest logit
    ).to(device)


model = build_model(ENCODER)

# Quick shape sanity check
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(device)
print("Output shape:", model(dummy).shape)   # expect (2, 1, IMG_SIZE, IMG_SIZE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Encoder: {ENCODER} | Total parameters: {n_params:,}")

## **8️⃣ Loss & Metrics**
smp ships ready-made loss functions. We combine two:

- **`smp.losses.DiceLoss`** → directly optimizes **overlap** between prediction and mask (robust to the background dominating).  
- **BCE-with-logits** → per-pixel classification loss.

We still track **Dice** and **IoU** ourselves for evaluation.

> 🔧 Our model outputs **raw logits** (no sigmoid inside). smp's `DiceLoss(mode="binary")` and
> `BCEWithLogitsLoss` both expect logits, so this lines up cleanly.

In [ ]:
def dice_coeff(preds, targets, eps=1e-7):
    """preds and targets are binary {0,1} tensors."""
    preds = preds.view(-1)
    targets = targets.view(-1)
    inter = (preds * targets).sum()
    return (2 * inter + eps) / (preds.sum() + targets.sum() + eps)


def iou_score(preds, targets, eps=1e-7):
    preds = preds.view(-1)
    targets = targets.view(-1)
    inter = (preds * targets).sum()
    union = preds.sum() + targets.sum() - inter
    return (inter + eps) / (union + eps)


class DiceBCELoss(nn.Module):
    """smp DiceLoss + BCEWithLogits (both operate on raw logits)."""
    def __init__(self):
        super().__init__()
        self.dice = smp.losses.DiceLoss(mode="binary", from_logits=True)
        self.bce  = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        return self.dice(logits, targets) + self.bce(logits, targets)


criterion = DiceBCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

## **9️⃣ Training the Model**
Each epoch we run a **training pass** (update weights) and a **validation pass** (measure Dice / IoU).

> ⏱️ On GPU this trains comfortably. On CPU it will be slow — drop `IMG_SIZE` to 128 and use a subset if you're not on a GPU.

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    dice_total, iou_total, n = 0.0, 0.0, 0
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        logits = model(imgs)
        preds = (torch.sigmoid(logits) > 0.5).float()
        dice_total += dice_coeff(preds, masks).item() * imgs.size(0)
        iou_total  += iou_score(preds, masks).item()  * imgs.size(0)
        n += imgs.size(0)
    return dice_total / n, iou_total / n


NUM_EPOCHS = 15
history = {"train_loss": [], "val_dice": [], "val_iou": []}

for epoch in range(NUM_EPOCHS):
    model.train()
    running = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(device), masks.to(device)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, masks)
        loss.backward()
        optimizer.step()

        running += loss.item() * imgs.size(0)

    train_loss = running / len(train_loader.dataset)
    val_dice, val_iou = evaluate(model, val_loader)

    history["train_loss"].append(train_loss)
    history["val_dice"].append(val_dice)
    history["val_iou"].append(val_iou)

    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
          f"train_loss={train_loss:.4f} | val_Dice={val_dice:.4f} | val_IoU={val_iou:.4f}")

## **🔟 Training Curves**

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))

ax[0].plot(history["train_loss"], marker="o")
ax[0].set_title("Training Loss"); ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Loss")

ax[1].plot(history["val_dice"], marker="o", label="Dice")
ax[1].plot(history["val_iou"],  marker="s", label="IoU")
ax[1].set_title("Validation Metrics"); ax[1].set_xlabel("Epoch"); ax[1].legend()

plt.tight_layout()
plt.show()

## **1️⃣1️⃣ Inspecting Predictions**
The real test of a segmentation model is **looking at the masks**. Below we show, for a few
validation images: the **input**, the **ground-truth mask**, and the **model's prediction**.

In [ ]:
@torch.no_grad()
def show_predictions(model, dataset, n=4):
    model.eval()
    idxs = np.random.choice(len(dataset), n, replace=False)
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))

    for row, idx in enumerate(idxs):
        img, mask = dataset[idx]
        logits = model(img.unsqueeze(0).to(device))
        pred = (torch.sigmoid(logits) > 0.5).float().cpu().squeeze()

        axes[row, 0].imshow(img.permute(1, 2, 0).numpy())
        axes[row, 0].set_title("Input"); axes[row, 0].axis("off")

        axes[row, 1].imshow(mask.squeeze(), cmap="gray")
        axes[row, 1].set_title("Ground Truth"); axes[row, 1].axis("off")

        axes[row, 2].imshow(pred, cmap="gray")
        axes[row, 2].set_title("Prediction"); axes[row, 2].axis("off")

    plt.tight_layout()
    plt.show()


show_predictions(model, val_ds, n=4)

#### Bonus: overlay the predicted forest region on the original image 🌳

In [ ]:
@torch.no_grad()
def show_overlay(model, dataset, n=3):
    model.eval()
    idxs = np.random.choice(len(dataset), n, replace=False)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 6))
    if n == 1:
        axes = [axes]

    for ax, idx in zip(axes, idxs):
        img, _ = dataset[idx]
        logits = model(img.unsqueeze(0).to(device))
        pred = (torch.sigmoid(logits) > 0.5).float().cpu().squeeze().numpy()

        base = img.permute(1, 2, 0).numpy().copy()
        # Tint predicted forest pixels green
        overlay = base.copy()
        overlay[pred == 1] = [0.1, 0.8, 0.2]
        blended = 0.6 * base + 0.4 * overlay

        ax.imshow(blended)
        ax.set_title("Predicted forest (green)")
        ax.axis("off")

    plt.tight_layout()
    plt.show()


show_overlay(model, val_ds, n=3)

## **1️⃣2️⃣ (Optional) Compare Different Encoders**
Because smp makes the encoder a one-word swap, we can **benchmark several backbones** on a
**short training run** and compare validation Dice. Below we train each for just a few epochs —
enough to see the trend without waiting forever.

> ⏳ This cell trains multiple models, so it's the slow one. Reduce `ENCODERS` or `QUICK_EPOCHS`
> if you're short on GPU time, or just skip it.

In [ ]:
ENCODERS = ["mobilenet_v2", "resnet18", "resnet34", "efficientnet-b0"]
QUICK_EPOCHS = 3

def quick_train(encoder_name, epochs=QUICK_EPOCHS):
    m = build_model(encoder_name)
    opt = torch.optim.Adam(m.parameters(), lr=1e-4)
    loss_fn = DiceBCELoss()
    for _ in range(epochs):
        m.train()
        for imgs, masks in train_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            opt.zero_grad()
            loss = loss_fn(m(imgs), masks)
            loss.backward()
            opt.step()
    dice, iou = evaluate(m, val_loader)
    n_params = sum(p.numel() for p in m.parameters())
    return dice, iou, n_params


results = {}
for enc in ENCODERS:
    dice, iou, n_params = quick_train(enc)
    results[enc] = (dice, iou, n_params)
    print(f"{enc:18s} | params={n_params/1e6:5.1f}M | val_Dice={dice:.4f} | val_IoU={iou:.4f}")

# Bar chart of val Dice per encoder
names = list(results.keys())
dices = [results[n][0] for n in names]
plt.figure(figsize=(8, 4))
plt.bar(names, dices, color="seagreen")
plt.ylabel("Validation Dice"); plt.title(f"Encoder comparison ({QUICK_EPOCHS} epochs each)")
plt.xticks(rotation=20); plt.ylim(0, 1)
for i, d in enumerate(dices):
    plt.text(i, d + 0.01, f"{d:.3f}", ha="center")
plt.tight_layout()
plt.show()

## **1️⃣3️⃣ Save the Model**

In [ ]:
torch.save(model.state_dict(), "unet_forest.pth")
print("✅ Saved to unet_forest.pth")

---
## **🎯 Bonus Challenges**
- 📈 **Add augmentation** (random flips/rotations) to the training set and see if val Dice improves.  
- 🏆 From the encoder comparison, **fully train the winner** (more epochs) and compare to `resnet34`.  
- 🧱 Try a **different decoder architecture** — smp also offers `smp.UnetPlusPlus`, `smp.FPN`, `smp.DeepLabV3Plus`. Same API, one-word swap.  
- ⚖️ Try **Dice-only** vs **BCE-only** vs the **combined** loss and compare.  
- ⏱️ Add a **learning-rate scheduler** (`CosineAnnealingLR`) and **early stopping**.  
- 🔍 Compute **per-image IoU** and look at the **worst** predictions — what kinds of scenes fail?  

---
### 🌲 **Now you have a working U-Net segmentation pipeline (with swappable encoders) for aerial forest imagery!**